In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots
from diffusion_geometry import Function, VectorField, Form
from generate_data import gen_2d_data
from diffusion_geometry import DiffusionGeometry




def sample_perturbed_grid(range_x, range_y, num_x, num_y, noise_strength=0.3):
    x_lin = np.linspace(range_x[0], range_x[1], num_x)
    y_lin = np.linspace(range_y[0], range_y[1], num_y)
    x_grid, y_grid = np.meshgrid(x_lin, y_lin)
    data = np.column_stack([x_grid.ravel(), y_grid.ravel()])
    data += noise_strength * np.random.randn(*data.shape)
    return data

def create_figure_1(fig, row):
    num_x = 20
    num_y = 20 
    lim = -4
    data = sample_perturbed_grid(range_x=(-lim,lim), range_y=(-lim,lim), num_x=num_x, num_y=num_y, noise_strength=0.1)

    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=100)

    X_pointwise = np.zeros((dg.n, 2))
    X_pointwise[:, 1] = data[:, 0]
    X = VectorField.from_pointwise_basis(X_pointwise, dg)

    Y_pointwise = np.zeros((dg.n, 2))
    Y_pointwise[:, 0] = data[:, 1]
    Y = VectorField.from_pointwise_basis(Y_pointwise, dg)

    Lie_XY = dg.lie_bracket(X, Y)

    real_XY_coeffs = np.zeros((dg.n, 2))
    real_XY_coeffs[:, 0] = data[:, 0]
    real_XY_coeffs[:, 1] = -data[:, 1]
    real_XY = VectorField.from_pointwise_basis(real_XY_coeffs, dg)

    scale = 0.12 
    line_width = 1

    fig1 = plot_quiver_2d(data, X.to_ambient(), scale=scale, line_width=line_width)
    fig2 = plot_quiver_2d(data, Y.to_ambient(), scale=scale, line_width=line_width)
    fig3 = plot_quiver_2d(data, Lie_XY.to_ambient(), scale=scale, line_width=line_width)
    # fig4 = plot_quiver_2d(data, (real_XY-Lie_XY).to_ambient(), scale=scale, line_width=line_width)



    fig.add_traces(fig1.data, rows=row, cols=1)
    fig.add_traces(fig2.data, rows=row, cols=2)
    fig.add_traces(fig3.data, rows=row, cols=3)
    # fig.add_traces(fig4.data, rows=row, cols=4)

    lim = 1
    for c in range(1,4):
        fig.update_xaxes(range=[-lim, lim], row=row, col=c)
        fig.update_yaxes(range=[-lim, lim], row=row, col=c)



def sample_circle(c, r, n=1):
    """
    Sample n points uniformly along a circle in 2D.
    c : center (2-vector)
    r : radius
    n : number of samples
    """
    theta = np.random.uniform(0, 2*np.pi, n)
    x = c[0] + r * np.cos(theta)
    y = c[1] + r * np.sin(theta)
    return np.column_stack((x, y))


def sample_grid(c, r, n):
    """
    Create a regular grid in the bounding box of a disk
    and keep only the points that lie inside the disk.

    c : center (2-vector)
    r : radius
    num_x, num_y : number of grid points in x and y directions
    """
    cx, cy = c
    x_lin = np.linspace(cx - r, cx + r, n)
    y_lin = np.linspace(cy - r, cy + r, n)

    X, Y = np.meshgrid(x_lin, y_lin)
    pts = np.column_stack([X.ravel(), Y.ravel()])

    # keep points with (x - cx)^2 + (y - cy)^2 <= r^2
    mask = (pts[:,0] - cx)**2 + (pts[:,1] - cy)**2 <= r**2
    return pts[mask]


def sample_disk(c, r, n=1):
    """
    Sample n points uniformly from inside a disk in 2D.
    c : center (2-vector)
    r : radius
    n : number of samples
    """
    theta = np.random.uniform(0, 2*np.pi, n)
    # sqrt(U) gives correct radial distribution
    rad = r * np.sqrt(np.random.uniform(0, 1, n))
    x = c[0] + rad * np.cos(theta)
    y = c[1] + rad * np.sin(theta)
    return np.column_stack((x, y))




def create_figure_2(fig, row):
    data1 = sample_grid(c=[0,0], r=2, n=20)
    data2 = sample_circle(c=[3,0], r=2, n=50)
    data3 = sample_circle(c=[-3,0], r=2, n=50)
    data4= sample_circle(c=[0,3], r=2, n=50)
    data5 = sample_circle(c=[0,-3], r=2, n=50)

    data = np.vstack([data1, data2, data3, data4, data5])

     # add uniform noise
    data += 0.2 * np.random.randn(*data.shape)

    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=100)

    dx_pointwise = np.zeros((dg.n, 2))
    dx_pointwise[:, 0] = 5.0
    dx = VectorField.from_pointwise_basis(dx_pointwise, dg)

    dy_pointwise = np.zeros((dg.n, 2))
    dy_pointwise[:, 1] = 5.0
    dy = VectorField.from_pointwise_basis(dy_pointwise, dg)

    Lie_dxdy = dg.lie_bracket(dx, dy)




    fig1 = plot_quiver_2d(data, dx.to_ambient(), scale=0.1, line_width=1)
    fig2 = plot_quiver_2d(data, dy.to_ambient(), scale=0.1, line_width=1)
    fig3 = plot_quiver_2d(data, Lie_dxdy.to_ambient(), scale=0.01, line_width=1)

    fig.add_traces(fig1.data, rows=row, cols=1)
    fig.add_traces(fig2.data, rows=row, cols=2)
    fig.add_traces(fig3.data, rows=row, cols=3)

    lim = 5.6
    for c in range(1,4):
        fig.update_xaxes(range=[-lim, lim], row=row, col=c)
        fig.update_yaxes(range=[-lim, lim], row=row, col=c)


def sample_fibonacci_sphere(n: int, radius: float = 1.0) -> np.ndarray:
    indices = np.arange(n)
    phi = np.pi * (3. - np.sqrt(5.))  # golden angle

    y = 1 - 2 * indices / (n - 1)
    r = np.sqrt(1 - y * y)
    theta = phi * indices

    x = np.cos(theta) * r
    z = np.sin(theta) * r

    points = radius * np.column_stack((x, y, z))
    return points


def spherical_frame(points: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Normalize points to unit sphere
    n = points / np.linalg.norm(points, axis=1, keepdims=True)
    x, y, z = n[:, 0], n[:, 1], n[:, 2]

    # e_phi = derivative wrt azimuthal angle φ
    e_phi = np.column_stack((-y, x, np.zeros_like(z)))
    norm_phi = np.linalg.norm(e_phi, axis=1, keepdims=True)
    # handle poles robustly
    norm_phi[norm_phi[:, 0] == 0] = 1.0
    e_phi /= norm_phi

    # e_theta = e_phi × n (right-handed frame)
    e_theta = np.cross(e_phi, n)

    return e_theta, e_phi, n


def create_figure_3(fig, row):
    data = sample_fibonacci_sphere(n=500, radius=3.0)

    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=100)

    e_theta, e_phi, _ = spherical_frame(data)

    E_theta = VectorField.from_pointwise_basis(e_theta, dg)
    E_phi = VectorField.from_pointwise_basis(e_phi, dg)

    Lie_Etheta_Ephi = dg.lie_bracket(E_theta, E_phi)

    scale = 0.3
    line_width = 1

    fig1 = plot_quiver_3d(data, E_theta.to_ambient(), scale=scale, line_width=line_width)
    fig2 = plot_quiver_3d(data, E_phi.to_ambient(), scale=scale, line_width=line_width)
    fig3 = plot_quiver_3d(data, Lie_Etheta_Ephi.to_ambient(), arrow_scale=0.045, line_width=0.8,scale=0.8)

    fig.add_traces(fig1.data, rows=row, cols=1)
    fig.add_traces(fig2.data, rows=row, cols=2)
    fig.add_traces(fig3.data, rows=row, cols=3)



num_rows = 3
num_cols = 3

specs = specs = [
    [{"type": "xy"}]*num_cols,  
    [{"type": "xy"}]*num_cols, 
    # [{"type": "xy"}]*num_cols,    
    [{"type": "scene"}]*num_cols,  
    # [{"type": "xy"}]*num_cols,  
    # [{"type": "scene"}]*num_cols,  
]

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    horizontal_spacing=0.001,
    vertical_spacing=0.05,
    shared_xaxes=True,
    shared_yaxes=True
)

create_figure_1(fig, row=1)
create_figure_2(fig, row=2)
create_figure_3(fig, row=3)


camera = dict(
    eye=dict(x=0, y=1.3, z=0),  
    up=dict(x=0, y=0, z=1),     
    center=dict(x=0, y=0, z=0)
)


for i in range(1, num_rows * num_cols + 1):
    scene_key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        scene_key: dict(
            camera=camera,
            aspectmode="data",
        )
    })


for r in range(num_rows):
    for c in range(num_cols):
        fig.update_xaxes(scaleanchor=f"y{c+1}", scaleratio=1, row=r+1, col=c+1)
        fig.update_yaxes(scaleanchor=f"x{c+1}", scaleratio=1, row=r+1, col=c+1)


clean_fig(fig)

aspect_ratio = 0.87
scale = 1300
fig.update_layout(width=scale * aspect_ratio, height=scale) 


fig.show()
fig.write_image("figs/lie_bracket.pdf", scale=1)





In [ ]:
label_list = [[r"X_1=x\partial_y", r"Y_1=y\partial_x", r"[X_1, Y_1]=x\partial_x-y\partial_y"],
              [r"X_2=\partial_x", r"Y_2=\partial_y", r"[X_2, Y_2]=0"],
              [r"e_\theta= \partial_\theta", r"e_\phi=\frac{1}{\sin(\theta)}\partial_\phi", r"[e_\theta, e_\phi]=-\mathrm{cot}(\theta)"],
             ]

for r in range(len(label_list)):
    for c in range(len(label_list[r])):
        label_list[r][c] = "$" + label_list[r][c] + "$"

def labels(row, col):
    return label_list[row-1][col-1]

overpic_labels(fig, labels, 
                       stretch_x=0.97,
                       stretch_y=1.03,
                       offset_y=15)